In [0]:
%sql
USE CATALOG nasa_dw;

CREATE TABLE IF NOT EXISTS silver.neo_clean
USING DELTA
COMMENT 'Datos de asteroides NEO limpios y enriquecidos'
AS SELECT * FROM nasa_dw.bronze.neo_raw WHERE 1=0;

MERGE INTO nasa_dw.silver.neo_clean AS tgt
USING (
    SELECT
        asteroide_id,
        nombre,
        CAST(fecha_acercamiento AS DATE)                        AS fecha_acercamiento,
        ROUND((diametro_min_km + diametro_max_km) / 2, 4)      AS diametro_promedio_km,
        COALESCE(es_potencialmente_peligroso, false)            AS es_potencialmente_peligroso,
        ROUND(distancia_tierra_km, 2)                           AS distancia_tierra_km,
        ROUND(distancia_tierra_km / 384400, 4)                  AS distancia_tierra_lu,
        ROUND(velocidad_kmh, 2)                                 AS velocidad_kmh,
        ROUND(velocidad_kmh / 3600, 4)                          AS velocidad_kms,
        magnitud_absoluta,
        CASE
            WHEN (diametro_min_km + diametro_max_km) / 2 < 0.05  THEN 'Pequeño'
            WHEN (diametro_min_km + diametro_max_km) / 2 < 0.5   THEN 'Mediano'
            WHEN (diametro_min_km + diametro_max_km) / 2 < 2.0   THEN 'Grande'
            ELSE 'Muy grande'
        END                                                     AS categoria_tamano,
        cuerpo_orbitado,
        url_nasa,
        CURRENT_TIMESTAMP()                                     AS fecha_procesamiento
    FROM nasa_dw.bronze.neo_raw
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY asteroide_id, fecha_acercamiento
        ORDER BY ingestion_timestamp DESC
    ) = 1
) AS src
ON tgt.asteroide_id = src.asteroide_id
AND tgt.fecha_acercamiento = src.fecha_acercamiento
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
-- Validamos lo que acomodamos.
SELECT
    categoria_tamano,
    COUNT(*)                            AS cantidad,
    ROUND(AVG(distancia_tierra_km), 0)  AS dist_promedio_km,
    ROUND(AVG(velocidad_kmh), 0)        AS vel_promedio_kmh
FROM nasa_dw.silver.neo_clean
GROUP BY categoria_tamano
ORDER BY cantidad DESC;